
# 06 — Génération de données avec Faker et tests avec Pytest

## Objectifs

- Générer des données synthétiques avec Faker.
- Créer un jeu de données reproductible.
- Simuler des défauts dans les données.
- Écrire des fonctions de nettoyage.
- Tester automatiquement les fonctions avec pytest.


In [1]:

import pandas as pd
import numpy as np
from faker import Faker

# Fixer les graines pour obtenir des résultats reproductibles
fake = Faker("fr_FR")
Faker.seed(42)
rng = np.random.default_rng(42)

# Générer 100 commandes fictives
n = 100

df = pd.DataFrame({
    "commande_id": range(1, n + 1),
    "client": [fake.name() for _ in range(n)],
    "ville": [fake.random_element(
        elements=["Dakar", "Thiès", "Saint-Louis", "Kaolack"]
    ) for _ in range(n)],
    "produit": [fake.random_element(
        elements=["Riz", "Mil", "Huile", "Sucre"]
    ) for _ in range(n)],
    "quantite": rng.integers(1, 11, size=n),
    "prix_unitaire": rng.integers(500, 5001, size=n)
})

df["montant"] = df["quantite"] * df["prix_unitaire"]

display(df.head())
print("Nombre de commandes :", len(df))


,commande_id,client,ville,produit,quantite,prix_unitaire,montant
0,1,Alexandre Traore,Dakar,Sucre,1,4248,4248
1,2,Gérard-Frédéric Joly,Saint-Louis,Riz,8,1399,11192
2,3,Yves Traore,Dakar,Huile,7,4122,28854
3,4,Aurore Maury-Briand,Thiès,Huile,5,533,2665
4,5,Alexandre Morin,Saint-Louis,Riz,5,4086,20430


Nombre de commandes : 100


In [2]:

# Créer une copie pour préserver les données originales
df_defauts = df.copy()

# Introduire des valeurs manquantes
df_defauts.loc[[2, 8, 15], "ville"] = None

# Introduire des quantités invalides
df_defauts.loc[5, "quantite"] = -3

# Introduire un doublon
df_defauts = pd.concat(
    [df_defauts, df_defauts.iloc[[0]]],
    ignore_index=True
)

# Examiner les défauts
print("Valeurs manquantes :")
print(df_defauts.isna().sum())

print("\nDoublons :", df_defauts.duplicated().sum())
print("\nQuantités négatives :",
      (df_defauts["quantite"] < 0).sum())


Valeurs manquantes :
commande_id      0
client           0
ville            3
produit          0
quantite         0
prix_unitaire    0
montant          0
dtype: int64

Doublons : 1

Quantités négatives : 1


In [3]:

def nettoyer_commandes(data):
    """Nettoyer un DataFrame de commandes commerciales."""
    resultat = data.copy()

    # Supprimer les doublons
    resultat = resultat.drop_duplicates()

    # Supprimer les lignes avec des champs essentiels manquants
    resultat = resultat.dropna(
        subset=["commande_id", "ville", "quantite", "prix_unitaire"]
    )

    # Supprimer les quantités et prix invalides
    resultat = resultat[
        (resultat["quantite"] > 0)
        & (resultat["prix_unitaire"] > 0)
    ]

    # Recalculer le montant pour garantir sa cohérence
    resultat["montant"] = (
        resultat["quantite"] * resultat["prix_unitaire"]
    )

    return resultat.reset_index(drop=True)


df_propre = nettoyer_commandes(df_defauts)

print("Avant :", len(df_defauts), "lignes")
print("Après :", len(df_propre), "lignes")
display(df_propre.head())


Avant : 101 lignes
Après : 96 lignes


,commande_id,client,ville,produit,quantite,prix_unitaire,montant
0,1,Alexandre Traore,Dakar,Sucre,1,4248,4248
1,2,Gérard-Frédéric Joly,Saint-Louis,Riz,8,1399,11192
2,4,Aurore Maury-Briand,Thiès,Huile,5,533,2665
3,5,Alexandre Morin,Saint-Louis,Riz,5,4086,20430
4,7,Victor du Vasseur,Dakar,Huile,1,4012,4012


In [4]:

from pathlib import Path

# Créer un fichier de tests
contenu_tests = '''
import pandas as pd
from nettoyage import nettoyer_commandes


def test_supprime_doublons():
    df = pd.DataFrame({
        "commande_id": [1, 1],
        "ville": ["Dakar", "Dakar"],
        "quantite": [2, 2],
        "prix_unitaire": [1000, 1000],
        "montant": [2000, 2000]
    })
    resultat = nettoyer_commandes(df)
    assert len(resultat) == 1


def test_supprime_quantites_negatives():
    df = pd.DataFrame({
        "commande_id": [1, 2],
        "ville": ["Dakar", "Thies"],
        "quantite": [2, -1],
        "prix_unitaire": [1000, 1000],
        "montant": [2000, -1000]
    })
    resultat = nettoyer_commandes(df)
    assert (resultat["quantite"] > 0).all()


def test_recalcule_montant():
    df = pd.DataFrame({
        "commande_id": [1],
        "ville": ["Dakar"],
        "quantite": [3],
        "prix_unitaire": [1000],
        "montant": [999]
    })
    resultat = nettoyer_commandes(df)
    assert resultat.iloc[0]["montant"] == 3000
'''

Path("test_nettoyage.py").write_text(contenu_tests)
print("Fichier de tests créé.")


Fichier de tests créé.



## Synthèse à retenir

- **Faker** génère des données synthétiques.
- **NumPy** génère des valeurs numériques.
- **Pandas** organise, contrôle et nettoie les données.
- **Une fonction de nettoyage** centralise les règles de qualité.
- **Pytest** vérifie automatiquement que ces règles sont respectées.

Dans un pipeline Data Engineering, ces compétences permettent de générer
des données de test, de détecter les défauts et de sécuriser les traitements.
